In [8]:
from pyspark.sql import SparkSession
import getpass

username = getpass.getuser()
spark = SparkSession. \
builder. \
config('spark.ui.port', '0'). \
config("spark.sql.warehouse.dir", f"/user/{username}/warehouse"). \
enableHiveSupport(). \
master('yarn'). \
getOrCreate()

# spark = SparkSession.builder.appName("Luyen tap choi choi spark").getOrCreate()

In [9]:
orders_rdd_all = spark.sparkContext.textFile("./orders_wh.csv")

In [10]:
#Take all, lay ca dong dau tien
orders_rdd_all.take(3)

['order_id,order_date,customer_id,order_status',
 '1,2013-07-25 00:00:00.0,11599,CLOSED',
 '2,2013-07-25 00:00:00.0,256,PENDING_PAYMENT']

In [33]:
header = orders_rdd_all.first()  # Lấy dòng đầu tiên làm header
orders_rdd = orders_rdd_all.filter(lambda line: line != header)  # Loại bỏ dòng header

In [34]:
orders_rdd.count()

68883

In [35]:
orders_rdd.take(10)

['1,2013-07-25 00:00:00.0,11599,CLOSED',
 '2,2013-07-25 00:00:00.0,256,PENDING_PAYMENT',
 '3,2013-07-25 00:00:00.0,12111,COMPLETE',
 '4,2013-07-25 00:00:00.0,8827,CLOSED',
 '5,2013-07-25 00:00:00.0,11318,COMPLETE',
 '6,2013-07-25 00:00:00.0,7130,COMPLETE',
 '7,2013-07-25 00:00:00.0,4530,COMPLETE',
 '8,2013-07-25 00:00:00.0,2911,PROCESSING',
 '9,2013-07-25 00:00:00.0,5657,PENDING_PAYMENT',
 '10,2013-07-25 00:00:00.0,5648,PENDING_PAYMENT']

In [36]:
orders_rdd.take(20)

['1,2013-07-25 00:00:00.0,11599,CLOSED',
 '2,2013-07-25 00:00:00.0,256,PENDING_PAYMENT',
 '3,2013-07-25 00:00:00.0,12111,COMPLETE',
 '4,2013-07-25 00:00:00.0,8827,CLOSED',
 '5,2013-07-25 00:00:00.0,11318,COMPLETE',
 '6,2013-07-25 00:00:00.0,7130,COMPLETE',
 '7,2013-07-25 00:00:00.0,4530,COMPLETE',
 '8,2013-07-25 00:00:00.0,2911,PROCESSING',
 '9,2013-07-25 00:00:00.0,5657,PENDING_PAYMENT',
 '10,2013-07-25 00:00:00.0,5648,PENDING_PAYMENT',
 '11,2013-07-25 00:00:00.0,918,PAYMENT_REVIEW',
 '12,2013-07-25 00:00:00.0,1837,CLOSED',
 '13,2013-07-25 00:00:00.0,9149,PENDING_PAYMENT',
 '14,2013-07-25 00:00:00.0,9842,PROCESSING',
 '15,2013-07-25 00:00:00.0,2568,COMPLETE',
 '16,2013-07-25 00:00:00.0,7276,PENDING_PAYMENT',
 '17,2013-07-25 00:00:00.0,2667,COMPLETE',
 '18,2013-07-25 00:00:00.0,1205,CLOSED',
 '19,2013-07-25 00:00:00.0,9488,PENDING_PAYMENT',
 '20,2013-07-25 00:00:00.0,9198,PROCESSING']

1: Đếm tổng số status

Output: ex- CLOSE, 100; COMPLETE, 200; ...
Sử dụng reduceByKey

In [37]:
map_rdd = orders_rdd.map(lambda x: (x.split(",")[3], 1))

In [38]:
map_rdd.take(10)

[('CLOSED', 1),
 ('PENDING_PAYMENT', 1),
 ('COMPLETE', 1),
 ('CLOSED', 1),
 ('COMPLETE', 1),
 ('COMPLETE', 1),
 ('COMPLETE', 1),
 ('PROCESSING', 1),
 ('PENDING_PAYMENT', 1),
 ('PENDING_PAYMENT', 1)]

In [39]:
reduce_rdd = map_rdd.reduceByKey(lambda x, y : x + y)

In [40]:
reduce_rdd.collect()

[('CLOSED', 7556),
 ('CANCELED', 1428),
 ('PENDING_PAYMENT', 15030),
 ('COMPLETE', 22899),
 ('PROCESSING', 8275),
 ('PAYMENT_REVIEW', 729),
 ('PENDING', 7610),
 ('ON_HOLD', 3798),
 ('SUSPECTED_FRAUD', 1558)]

In [43]:
sort_with_count_asce = reduce_rdd.sortBy(lambda x: x[1])

In [44]:
sort_with_count.collect()

[('order_status', 1),
 ('PAYMENT_REVIEW', 729),
 ('CANCELED', 1428),
 ('SUSPECTED_FRAUD', 1558),
 ('ON_HOLD', 3798),
 ('CLOSED', 7556),
 ('PENDING', 7610),
 ('PROCESSING', 8275),
 ('PENDING_PAYMENT', 15030),
 ('COMPLETE', 22899)]

In [45]:
#Theo thứ tự giảm dần count:
sort_with_count_desc = reduce_rdd.sortBy(lambda x: x[1], False)
sort_with_count_desc.collect()

[('COMPLETE', 22899),
 ('PENDING_PAYMENT', 15030),
 ('PROCESSING', 8275),
 ('PENDING', 7610),
 ('CLOSED', 7556),
 ('ON_HOLD', 3798),
 ('SUSPECTED_FRAUD', 1558),
 ('CANCELED', 1428),
 ('PAYMENT_REVIEW', 729)]

2: Top 10 khách hàng order nhiều nhất

In [46]:
customer_mapped_rdd = orders_rdd.map(lambda x: (x.split(",")[2], 1))

In [47]:
customer_mapped_rdd.take(5)

[('11599', 1), ('256', 1), ('12111', 1), ('8827', 1), ('11318', 1)]

In [48]:
customer_reduce_rdd = customer_mapped_rdd.reduceByKey(lambda x, y: x+y)

In [49]:
customer_reduce_rdd.take(5)

[('256', 10), ('12111', 6), ('11318', 6), ('7130', 7), ('2911', 6)]

In [50]:
customer_sort_10_rdd = customer_reduce_rdd.sortBy(lambda x: x[1], False)

In [51]:
customer_sort_10_rdd.take(10)

[('5897', 16),
 ('6316', 16),
 ('12431', 16),
 ('569', 16),
 ('4320', 15),
 ('221', 15),
 ('5624', 15),
 ('5283', 15),
 ('12284', 15),
 ('5654', 15)]

3: Tổng số lượng khách hàng là bao nhiêu?

In [53]:
cust_rdd = orders_rdd.map(lambda x: x.split(",")[2])

In [54]:
cust_rdd.take(5)

['11599', '256', '12111', '8827', '11318']

In [55]:
total_cust = cust_rdd.distinct()

In [58]:
total_cust.count()

12405

4: Khách hàng có số lượng có order với status là CLOSED là nhiều nhất

In [59]:
filtered_rdd = orders_rdd.filter(lambda x: x.split(",")[3] == "CLOSED")

In [60]:
filtered_rdd.take(5)

['1,2013-07-25 00:00:00.0,11599,CLOSED',
 '4,2013-07-25 00:00:00.0,8827,CLOSED',
 '12,2013-07-25 00:00:00.0,1837,CLOSED',
 '18,2013-07-25 00:00:00.0,1205,CLOSED',
 '24,2013-07-25 00:00:00.0,11441,CLOSED']

In [62]:
cust_closed = filtered_rdd.map(lambda x: (x.split(",")[2], 1))

In [63]:
cust_closed.take(5)

[('11599', 1), ('8827', 1), ('1837', 1), ('1205', 1), ('11441', 1)]

In [64]:
cust_closed_reduced = cust_closed.reduceByKey(lambda x, y: x+y)

In [65]:
cust_closed_reduced.take(5)

[('5863', 1), ('12271', 2), ('7073', 1), ('3065', 2), ('5116', 2)]

In [66]:
sorted_cust_closed = cust_closed_reduced.sortBy(lambda x: x[1], False)

In [67]:
#Id khach hang co luong order nhieu nhat
sorted_cust_closed.take(1)

[('1833', 6)]